### continuation de la carto en suivant l'exercice 3-2-2-strategie-manquants

reprise des requetes permettant la génération du dataframe

In [1]:
import duckdb
import pandas as pd
import pyarrow as py
import matplotlib.pyplot as plt

con = duckdb.connect()

In [2]:
con.sql("DESCRIBE SELECT * FROM '../../data/food.parquet'").show(max_rows=200)

┌────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│              column_name               │                                                                                                                       column_type                                                                                                                       │  null   │   key   │ default │  extra  │
│                varchar                 │                                                                                                                         varchar                                                                                                                         │ varchar │ varchar │ varchar │ varchar │
├

Ici on créer la table product nutrition, qui sera similaire au parquet pour la granularité : une ligne par produit cependant ce schéma diffère en trois points :

Les nutriments sont aplatis. Le parquet les stocke dans une liste de structures par produit, et la table les place chacun dans sa propre colonne.
Les listes sont réduites à leur premier élément pour product_name, ingredients_text et food_groups_tag_1. Les autres langues et groupes sont perdus.
Seules les colonnes utiles sont gardées, avec quelques transformations (NULLIF pour brands pour gérer les chaines vides, array_to_string pour les allergènes pour casser la liste et récupéré des strings) et list_filter(...'nutriment')[1] permet de récupérer la 1er valeur de ces structure

In [3]:
con.sql("""
    CREATE TABLE products_nutrition AS
    SELECT
        t.code,
        t.product_name[1].text AS product_name,
        NULLIF(t.brands, '') AS brands,
        t.quantity,
        t.categories_tags,
        t.compared_to_category AS main_category,
        t.food_groups_tags[1] AS food_groups_tag_1,
        t.countries_tags,
        t.labels_tags,
        t.stores,
        array_to_string(t.allergens_tags, ',') AS allergens,
        t.ingredients_text[1].text AS ingredients_text,
        t.nova_group,
        t.nutriscore_score,
        t.nutriscore_grade,
        t.environmental_score_grade,
        t.serving_size,
        t.completeness,
        list_filter(t.nutriments, x -> x.name = 'energy-kcal')[1]."100g" AS energy_kcal_100g,
        COALESCE(
            list_filter(t.nutriments, x -> x.name = 'energy-kj')[1]."100g",
            list_filter(t.nutriments, x -> x.name = 'energy')[1]."100g"
        ) AS energy_kj_100g,
        list_filter(t.nutriments, x -> x.name = 'fat')[1]."100g" AS fat_100g,
        list_filter(t.nutriments, x -> x.name = 'saturated-fat')[1]."100g" AS saturated_fat_100g,
        list_filter(t.nutriments, x -> x.name = 'carbohydrates')[1]."100g" AS carbohydrates_100g,
        list_filter(t.nutriments, x -> x.name = 'sugars')[1]."100g" AS sugars_100g,
        list_filter(t.nutriments, x -> x.name = 'fiber')[1]."100g" AS fiber_100g,
        list_filter(t.nutriments, x -> x.name = 'proteins')[1]."100g" AS proteins_100g,
        list_filter(t.nutriments, x -> x.name = 'salt')[1]."100g" AS salt_100g,
        list_filter(t.nutriments, x -> x.name = 'sodium')[1]."100g" AS sodium_100g,
        list_filter(t.nutriments, x -> x.name = 'fruits-vegetables-legumes')[1]."100g" AS fruits_vegetables_legumes_100g
    FROM '../../data/food.parquet' AS t
""")

In [53]:
con.sql("DROP TABLE products_nutrition")

In [4]:
df = con.sql("SELECT * FROM products_nutrition").df()

### Stratégie de valeurs manquantes par colonne et par usage

| colonne | usage | manquants | décision | pourquoi |
|---|---|---|---|---|
| allergens | affichage, assistant | 72 % | NULL, ou chaîne vide sans ingrédients → NA + drapeau allergens_missing ; chaîne vide avec ingrédients → « aucun allergène déclaré » ; jamais imputer | sans ingrédients, l’absence d’allergène ne peut pas être affirmée (manque quasi identique à ingredients_text) ; un allergène inventé ou omis est un risque pour l’utilisateur, l’assistant répond « information non disponible » |
| brands | affichage, substitution (même marque), assistant | 34 % | garder NA + drapeau ; « unknown » à l'affichage seulement | aucune valeur plausible à inventer |
| product_name | affichage, assistant | 6 % | garder NA + drapeau product_name_missing ; « nom non renseigné » à l’affichage seulement ; jamais imputer | aucun nom plausible à inventer ; sans nom, le produit reste utilisable pour le score et la substitution ; seul le premier nom de la liste (première langue) est conservé |
| categories_tags | substitution, features ML (food_groups_tag_1), assistant | 56 % | règle métier : inclassable si vide (NULL ou liste vide) et food_groups_tag_1 NULL → drapeau unclassifiable, exclu de la substitution, ligne conservée ; jamais imputée | sans catégorie ni groupe alimentaire, impossible de comparer un produit à un « même rayon » ; la ligne reste utile pour le Nutri-Score, l’affichage et l’assistant, donc pas de suppression |
| completness | analyse interne (justification du MAR), features ML | 0 % | garder telle quelle, non exposée dans l’application | mesure le remplissage de la fiche, partiellement redondante avec les drapeaux _missing ; valeurs jusqu’à 1,1 (0,36 % > 1), conservées car elles ne sont pas aberrantes |
| ingredients_text | assistant, features texte | 72 % | garder NA + drapeau | aucune valeur plausible à inventer |
| nova_group | affichage | 75 % | garder NA + drapeau | dérivée des ingrédients, si ingredient absent = 98.5% de chance que nova absent aussi |
| nutriscore_score/_grade | cible du modèle, affichage | 34 % | garder ; « unknown » → NA ; jamais imputer | une cible imputée est une cible inventée |
| energy_kcal_100g | score, affichage, features ML, substitution, assistant | 28 % | imputation métier : kJ / 4,184, sinon 4/4/9 ; reste → médiane par food_groups_tag_1 calculée sur le train uniquement, après le split + drapeau | dérivable avant d'être devinée, attention ici le drapeau missing ne veut pas dire valeur manquante, il veut dire imputer par médiane après tentative de derivation |
| sugars_100g | score, features ML, affichage, assistant | 32 % | garder NA + drapeau sugars_missing, jamais imputer | une valeur inventée fausserait le score, manque MAR lié à la complétude de la fiche et au catégorie de nourriture |
| salt_100g / sodium_100g | score, features ML, substitution, affichage, assistant | 37 % | imputation métier : sel = sodium × 2,5 (et inverse) | relation exacte, zéro risque |
| fiber_100g | score, features ML, substitution, affichage, assistant | 62 % | garder NA + drapeau fiber_missing + jamais imputer | MNAR : absence = souvent « pas de fibres », pas d'invention de valeur pour une composante du nutriscore |
| fruits_vegetables_legumes_100g | score, features ML, subsitution, affichage, assistant | 83 % | NA + drapeau, jamais imputer, affichage"non renseigné" | a limité aux food_groups_tag_1 si elle est indiqué |
| serving_size | sel par portion (affichage) | 68 % | a garder, jamais imputé, affichage des valeurs pour 100g si NA | la portion est propre à chaque produit, utile quand présente, jamais imputée |
| fat_100g | features ML, contrôle de cohérence, substitution, affichage, assistant | 28 % | garder NA + drapeau fat_missing ; reste → médiane par food_groups_tag_1 calculée sur le train uniquement, après le split | MAR : suit la complétude du tableau nutritionnel ; sert de borne haute pour les saturées, reconstitution testée, regle métier atwater testé : reconstitution du fat si 3 applicable à 0,16 % des lignes seulement, gain négligeable pour une approximation, on drop |
| proteins_100g | score, features ML, affichage, substitution, assistant | 28 % | garder NA + drapeau proteins_missing ; reste → médiane par food_groups_tag_1 calculée sur le train uniquement, après le split | MAR : manque groupé composante favorable du Nutri-Score |
| carbohydrates_100g | features ML, contrôle de cohérence, substitution, affichage, assistant | 28 % | garder NA + drapeau carbohydrates_missing ; valeurs > 100 → NA ; sert de borne haute pour sugars_100g ; jamais imputée | utile au profil nutritionnel ; une valeur inventée fausserait les contrôles de cohérence |
| saturated_fat_100g | score, features ML, affichage, substitution, assistant | 34 % | garder NA + drapeau saturated_fat_missing, règle métier : si fat_100g < 0,5 g → saturées = 0 ; sinon jamais imputer ; score non calculable si NA | composante directe du Nutri-Score, une valeur inventée fausserait le score |
| produit sans aucun nutriment clé | — | 27 % | supprimer la ligne | inutilisable pour le score et la substitution |

### Requetes DuckDB sur les différentes colonnes

#### Colonne Stores

In [35]:
# 1. Type de la colonne
con.sql("""
    SELECT column_name, column_type
    FROM (DESCRIBE products_nutrition)
    WHERE column_name = 'stores'
""").show()

# 2. Taux de remplissage
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE stores IS NULL) AS nulls,
        COUNT(*) FILTER (WHERE stores = '') AS empty_strings,
        COUNT(DISTINCT stores) AS distinct_values
    FROM products_nutrition
""").show()

# 3. Valeurs les plus fréquentes
con.sql("""
    SELECT stores, COUNT(*) AS n
    FROM products_nutrition
    WHERE stores IS NOT NULL AND stores <> ''
    GROUP BY stores
    ORDER BY n DESC
    LIMIT 30
""").show(max_rows=1000, max_width=200)

┌─────────────┬─────────────┐
│ column_name │ column_type │
│   varchar   │   varchar   │
├─────────────┼─────────────┤
│ stores      │ VARCHAR     │
└─────────────┴─────────────┘

┌─────────┬─────────┬───────────────┬─────────────────┐
│  total  │  nulls  │ empty_strings │ distinct_values │
│  int64  │  int64  │     int64     │      int64      │
├─────────┼─────────┼───────────────┼─────────────────┤
│ 4636471 │ 3635220 │        534924 │           23914 │
└─────────┴─────────┴───────────────┴─────────────────┘

┌─────────────────┬───────┐
│     stores      │   n   │
│     varchar     │ int64 │
├─────────────────┼───────┤
│ Carrefour       │ 18632 │
│ Lidl            │ 17545 │
│ Migros          │ 14337 │
│ Delhaize        │ 12686 │
│ Auchan          │ 12541 │
│ Tesco           │ 11172 │
│ Leclerc         │ 10747 │
│ Mercadona       │ 10108 │
│ Aldi            │  9765 │
│ Magasins U      │  9525 │
│ REWE            │  8328 │
│ bonÀrea         │  7565 │
│ Casino          │  7058 │
│ Walm

#### Colonne environmental_score_grade

In [39]:
# 1. Type de la colonne
con.sql("""
    SELECT column_name, column_type
    FROM (DESCRIBE products_nutrition)
    WHERE column_name = 'environmental_score_grade'
""").show()

# 2. Taux de remplissage
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE environmental_score_grade IS NULL) AS nulls,
        COUNT(*) FILTER (WHERE environmental_score_grade = '') AS empty_strings,
        COUNT(*) FILTER (WHERE lower(environmental_score_grade) = 'unknown') AS unknown,
        COUNT(*) FILTER (WHERE lower(environmental_score_grade) = 'not-applicable') AS not_applicable,
        COUNT(DISTINCT environmental_score_grade) AS distinct_values
    FROM products_nutrition
""").show()

# 3. Toutes les valeurs, avec leur fréquence
con.sql("""
    SELECT environmental_score_grade, COUNT(*) AS n
    FROM products_nutrition
    GROUP BY environmental_score_grade
    ORDER BY n DESC
""").show(max_rows=100)

┌───────────────────────────┬─────────────┐
│        column_name        │ column_type │
│          varchar          │   varchar   │
├───────────────────────────┼─────────────┤
│ environmental_score_grade │ VARCHAR     │
└───────────────────────────┴─────────────┘

┌─────────┬────────┬───────────────┬─────────┬────────────────┬─────────────────┐
│  total  │ nulls  │ empty_strings │ unknown │ not_applicable │ distinct_values │
│  int64  │ int64  │     int64     │  int64  │     int64      │      int64      │
├─────────┼────────┼───────────────┼─────────┼────────────────┼─────────────────┤
│ 4636471 │ 829385 │             0 │ 2876549 │          36806 │               9 │
└─────────┴────────┴───────────────┴─────────┴────────────────┴─────────────────┘

┌───────────────────────────┬─────────┐
│ environmental_score_grade │    n    │
│          varchar          │  int64  │
├───────────────────────────┼─────────┤
│ unknown                   │ 2876549 │
│ NULL                      │  829385 │
│ 

#### Colonne countries_tags

In [40]:
# 1. Type de la colonne
con.sql("""
    SELECT column_name, column_type
    FROM (DESCRIBE products_nutrition)
    WHERE column_name = 'countries_tags'
""").show()

# 2. Taux de remplissage (version liste VARCHAR[])
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE countries_tags IS NULL) AS nulls,
        COUNT(*) FILTER (WHERE len(countries_tags) = 0) AS empty_lists,
        COUNT(*) FILTER (WHERE list_contains(countries_tags, 'en:france')) AS with_france,
        COUNT(*) FILTER (WHERE len(countries_tags) > 1) AS several_countries
    FROM products_nutrition
""").show()

# 3. Pays les plus fréquents (une ligne par pays après UNNEST)
con.sql("""
    SELECT country, COUNT(*) AS n
    FROM (SELECT UNNEST(countries_tags) AS country FROM products_nutrition)
    GROUP BY country
    ORDER BY n DESC
    LIMIT 30
""").show(max_rows=50, max_width=200)

┌────────────────┬─────────────┐
│  column_name   │ column_type │
│    varchar     │   varchar   │
├────────────────┼─────────────┤
│ countries_tags │ VARCHAR[]   │
└────────────────┴─────────────┘

┌─────────┬───────┬─────────────┬─────────────┬───────────────────┐
│  total  │ nulls │ empty_lists │ with_france │ several_countries │
│  int64  │ int64 │    int64    │    int64    │       int64       │
├─────────┼───────┼─────────────┼─────────────┼───────────────────┤
│ 4636471 │  9741 │       13085 │     1247336 │            222531 │
└─────────┴───────┴─────────────┴─────────────┴───────────────────┘

┌───────────────────┬─────────┐
│      country      │    n    │
│      varchar      │  int64  │
├───────────────────┼─────────┤
│ en:france         │ 1247336 │
│ en:united-states  │  941310 │
│ en:germany        │  417492 │
│ en:spain          │  355835 │
│ en:italy          │  268371 │
│ en:united-kingdom │  190195 │
│ en:canada         │  122746 │
│ en:netherlands    │  106164 │
│ en:swi

#### Colonne allergens

In [44]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE allergens IS NULL) AS nulls,
        COUNT(*) FILTER (WHERE allergens = '') AS empty_strings,
        COUNT(*) FILTER (WHERE allergens <> '') AS with_allergens
    FROM products_nutrition
""").show()

┌─────────┬───────┬───────────────┬────────────────┐
│  total  │ nulls │ empty_strings │ with_allergens │
│  int64  │ int64 │     int64     │     int64      │
├─────────┼───────┼───────────────┼────────────────┤
│ 4636471 │ 43100 │       3843718 │         749653 │
└─────────┴───────┴───────────────┴────────────────┘



In [42]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE allergens = '' AND ingredients_text IS NULL) AS empty_no_ingredients,
        COUNT(*) FILTER (WHERE allergens = '' AND ingredients_text IS NOT NULL) AS empty_with_ingredients,
        COUNT(*) FILTER (WHERE allergens <> '' AND ingredients_text IS NULL) AS allergens_no_ingredients
    FROM products_nutrition
""").show()

┌──────────────────────┬────────────────────────┬──────────────────────────┐
│ empty_no_ingredients │ empty_with_ingredients │ allergens_no_ingredients │
│        int64         │         int64          │          int64           │
├──────────────────────┼────────────────────────┼──────────────────────────┤
│              3293384 │                 550334 │                     3392 │
└──────────────────────┴────────────────────────┴──────────────────────────┘



In [45]:
con.sql("""
    SELECT
        ROUND(100.0 * COUNT(*) FILTER (
            WHERE allergens IS NULL OR (allergens = '' AND ingredients_text IS NULL)
        ) / COUNT(*), 1) AS pct_unknown
    FROM products_nutrition
""").show()

┌─────────────┐
│ pct_unknown │
│   double    │
├─────────────┤
│        72.0 │
└─────────────┘



#### Colonne food_groups_tag_1, categories_tag, main_category

In [5]:
con.sql("""
    SELECT food_groups_tag_1, COUNT(*) AS n
    FROM products_nutrition
    WHERE food_groups_tag_1 IS NULL OR food_groups_tag_1 ILIKE '%unknown%'
    GROUP BY food_groups_tag_1
    ORDER BY n DESC
""").show()

┌───────────────────┬─────────┐
│ food_groups_tag_1 │    n    │
│      varchar      │  int64  │
├───────────────────┼─────────┤
│ NULL              │ 3175085 │
└───────────────────┴─────────┘



In [6]:
con.sql("""
    SELECT column_name, column_type
    FROM (DESCRIBE products_nutrition)
    WHERE column_name IN ('categories_tags', 'food_groups_tag_1', 'main_category')
""").show()

┌───────────────────┬─────────────┐
│    column_name    │ column_type │
│      varchar      │   varchar   │
├───────────────────┼─────────────┤
│ categories_tags   │ VARCHAR[]   │
│ main_category     │ VARCHAR     │
│ food_groups_tag_1 │ VARCHAR     │
└───────────────────┴─────────────┘



requete qui va vérifier : le nombre de null sur food_groups_tag_1, le nombre de null de categories_tags, le nombre de null ou main_category est null ou une chaine vide, le nombre de lignes ou fgt1 est null et categorie_tage null ou 0 et le nombre de ligne ou il n'y a aucune categ

In [50]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE food_groups_tag_1 IS NULL) AS food_groups_tag_1_null,
        COUNT(*) FILTER (WHERE categories_tags IS NULL OR len(categories_tags) = 0) AS categories_empty,
        COUNT(*) FILTER (WHERE main_category IS NULL OR trim(main_category) = '') AS main_category_empty,
        COUNT(*) FILTER (
            WHERE food_groups_tag_1 IS NULL
              AND (categories_tags IS NULL OR len(categories_tags) = 0)
        ) AS unclassifiable,
        COUNT(*) FILTER (
            WHERE food_groups_tag_1 IS NULL
              AND (categories_tags IS NULL OR len(categories_tags) = 0)
              AND (main_category IS NULL OR trim(main_category) = '')
        ) AS no_category_at_all
    FROM products_nutrition
""").show()

┌─────────┬────────────────────────┬──────────────────┬─────────────────────┬────────────────┬────────────────────┐
│  total  │ food_groups_tag_1_null │ categories_empty │ main_category_empty │ unclassifiable │ no_category_at_all │
│  int64  │         int64          │      int64       │        int64        │     int64      │       int64        │
├─────────┼────────────────────────┼──────────────────┼─────────────────────┼────────────────┼────────────────────┤
│ 4636471 │                3175085 │          2772014 │             2849448 │        2772014 │            2767662 │
└─────────┴────────────────────────┴──────────────────┴─────────────────────┴────────────────┴────────────────────┘



requete qui va vérifier les lignes ou categorie_tag n'est pas renseigné, et le cas ou il est n'est pas renseigné et ou est les nutriment clé ne sont pas renseigné

In [51]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE categories_tags IS NULL OR len(categories_tags) = 0) AS unclassifiable,
        COUNT(*) FILTER (
            WHERE (categories_tags IS NULL OR len(categories_tags) = 0)
              AND energy_kcal_100g IS NULL AND energy_kj_100g IS NULL
              AND saturated_fat_100g IS NULL AND sugars_100g IS NULL
              AND proteins_100g IS NULL AND salt_100g IS NULL AND sodium_100g IS NULL
        ) AS unclassifiable_and_no_nutrient
    FROM products_nutrition
""").show()

┌────────────────┬────────────────────────────────┐
│ unclassifiable │ unclassifiable_and_no_nutrient │
│     int64      │             int64              │
├────────────────┼────────────────────────────────┤
│        2772014 │                         963259 │
└────────────────┴────────────────────────────────┘



la requete montre le nombre de lignes si categorie tag est nulll et food_group_tag est non nul

In [52]:
con.sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE food_groups_tag_1 IS NULL
              AND categories_tags IS NOT NULL AND len(categories_tags) > 0
        ) AS no_food_group_with_categories
    FROM products_nutrition
""").show()

┌───────────────────────────────┐
│ no_food_group_with_categories │
│             int64             │
├───────────────────────────────┤
│                        403071 │
└───────────────────────────────┘



#### Colonne completness

In [46]:
con.sql("""
    SELECT
        ROUND(100.0 * COUNT(*) FILTER (WHERE completeness IS NULL) / COUNT(*), 1) AS pct_null,
        MIN(completeness) AS min_value,
        MAX(completeness) AS max_value,
        COUNT(*) FILTER (WHERE completeness < 0 OR completeness > 1) AS out_of_0_1
    FROM products_nutrition
""").show()

┌──────────┬───────────┬───────────┬────────────┐
│ pct_null │ min_value │ max_value │ out_of_0_1 │
│  double  │   float   │   float   │   int64    │
├──────────┼───────────┼───────────┼────────────┤
│      0.0 │       0.0 │       1.1 │      16674 │
└──────────┴───────────┴───────────┴────────────┘



#### Colonne product_name

In [47]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE product_name IS NULL) AS nulls,
        COUNT(*) FILTER (WHERE product_name = '') AS empty_strings,
        COUNT(*) FILTER (WHERE product_name <> '' AND trim(product_name) = '') AS spaces_only,
        ROUND(100.0 * COUNT(*) FILTER (
            WHERE product_name IS NULL OR trim(product_name) = ''
        ) / COUNT(*), 1) AS pct_missing
    FROM products_nutrition
""").show()

┌─────────┬────────┬───────────────┬─────────────┬─────────────┐
│  total  │ nulls  │ empty_strings │ spaces_only │ pct_missing │
│  int64  │ int64  │     int64     │    int64    │   double    │
├─────────┼────────┼───────────────┼─────────────┼─────────────┤
│ 4636471 │ 280136 │             0 │           1 │         6.0 │
└─────────┴────────┴───────────────┴─────────────┴─────────────┘



Après analyse des différentes colonnes je décide de rajouter product_name et completeness et allergens au tableau et d'enlever quantity, stores et environmental_score_grade,  countries_tags, labels_tags, le schéma sera modifié au prochaine exercice

### Rédaction des méthodes d'implémentation des décisions : 

Vérification si les unknown sont bien présent sur le parquet aussi

In [8]:
con.sql("""
    SELECT nutriscore_grade, COUNT(*) AS n
    FROM products_nutrition
    GROUP BY 1
    ORDER BY n DESC;
    """)

┌──────────────────┬─────────┐
│ nutriscore_grade │    n    │
│     varchar      │  int64  │
├──────────────────┼─────────┤
│ unknown          │ 3110742 │
│ e                │  387545 │
│ d                │  346120 │
│ c                │  287328 │
│ a                │  200871 │
│ b                │  159205 │
│ not-applicable   │  101213 │
│ NULL             │   43447 │
└──────────────────┴─────────┘

In [9]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE nutriscore_score IS NULL) AS score_null,
        ROUND(100.0 * COUNT(*) FILTER (WHERE nutriscore_score IS NULL) / COUNT(*), 1) AS pct_null
    FROM products_nutrition
""")

┌─────────┬────────────┬──────────┐
│  total  │ score_null │ pct_null │
│  int64  │   int64    │  double  │
├─────────┼────────────┼──────────┤
│ 4636471 │    3255402 │     70.2 │
└─────────┴────────────┴──────────┘

In [10]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE nutriscore_grade IS NULL) AS score_null,
        ROUND(100.0 * COUNT(*) FILTER (WHERE nutriscore_score IS NULL) / COUNT(*), 1) AS pct_null
    FROM products_nutrition
""")

┌─────────┬────────────┬──────────┐
│  total  │ score_null │ pct_null │
│  int64  │   int64    │  double  │
├─────────┼────────────┼──────────┤
│ 4636471 │      43447 │     70.2 │
└─────────┴────────────┴──────────┘

Vérification des valeurs aberrantes : 

Sel/sodium

In [11]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE salt_100g < 0 OR salt_100g > 100) AS salt_outlier,
        COUNT(*) FILTER (WHERE sodium_100g < 0 OR sodium_100g > 40) AS sodium_outlier
    FROM products_nutrition
""")

┌──────────────┬────────────────┐
│ salt_outlier │ sodium_outlier │
│    int64     │     int64      │
├──────────────┼────────────────┤
│         2543 │           2637 │
└──────────────┴────────────────┘

Glucides

In [12]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE carbohydrates_100g IS NULL) AS carbs_null,
        COUNT(*) FILTER (WHERE carbohydrates_100g > 100) AS carbs_above_100,
        COUNT(*) FILTER (WHERE carbohydrates_100g < 0) AS carbs_negative,
        ROUND(100.0 * COUNT(*) FILTER (WHERE carbohydrates_100g > 100) / COUNT(*), 3) AS pct_above_100
    FROM products_nutrition
""").show()

┌─────────┬────────────┬─────────────────┬────────────────┬───────────────┐
│  total  │ carbs_null │ carbs_above_100 │ carbs_negative │ pct_above_100 │
│  int64  │   int64    │      int64      │     int64      │    double     │
├─────────┼────────────┼─────────────────┼────────────────┼───────────────┤
│ 4636471 │    1318649 │           14334 │              1 │         0.309 │
└─────────┴────────────┴─────────────────┴────────────────┴───────────────┘



Avant de passer colonne par colonne, il serais intéréssant de neutraliser les valeurs aberrantes des colonnes a 100g avant tout chose, je vais rédiger une méthode générique qui imputera NA dès qu'une de ces valeurs seront détecté.

In [13]:
def neutralize_range(
    df: pd.DataFrame, column: str, low: float, high: float, flag: bool = True
) -> tuple[pd.DataFrame, dict[str, int]]:
    """Valeurs hors de [low, high] neutralisées en NA, jamais imputées."""
    res = df.copy()
    values = res[column]

    # Garde-fou : valeurs impossibles pour 100 g de produit -> NA
    invalid = values.notna() & ~values.between(low, high)
    values = values.mask(invalid)

    # Drapeau : NA d'origine + valeurs neutralisées
    if flag:
        res[f"{column}_missing"] = values.isna().astype("boolean")

    res[column] = values
    return res, {
        "invalid_to_na": int(invalid.sum()),
        "missing_total": int(values.isna().sum()),
    }

Grace au flag des paramètre de méthode, ça permettra d'indiquer aussi si on veux que NA soit imputer ou non, nécéssaire pour energy_kj qui sert surtout a construire energy_kcal

fonction de normalisation pour nutriscore_grade, ici on remplace toute les valeurs unknown ou not-applicable par des pd.Na
la fonction retourne un dataframe nettoyé et modifié (ajout d'une colonne qui indique les grade_non_applicable en true/false) ainsi que le nombre de lignes modifier en unknow et not applicable via un dictionnaire

In [48]:
def normalize_grade(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """1. Cible : « unknown » et « not-applicable » deviennent NA ; on ne remplit jamais."""
    res = df.copy()
    grade = res["nutriscore_grade"].astype("string").str.strip().str.lower()
    unknown = grade.eq("unknown").fillna(False)
    not_applicable = grade.eq("not-applicable").fillna(False)

    # Drapeau : distingue « non applicable » des autres manquants (NULL d'origine inclus)
    res["grade_not_applicable"] = not_applicable.astype("boolean")

    res["nutriscore_grade"] = res["nutriscore_grade"].mask(unknown | not_applicable, pd.NA)
    return res, {
        "unknown": int(unknown.sum()),
        "not_applicable": int(not_applicable.sum()),
    }

on passe a la méthode drapeau pour les colonnes dont les valeurs manquantes ne seront pas neutralisé par des méthodes personalisé : brands, ingredients_text et nova_group

In [21]:
def flag_missing(df: pd.DataFrame, column: str) -> tuple[pd.DataFrame, int]:
    """4. et 5. Fibres, marques : la valeur reste manquante, le modèle saura qu'elle manque."""
    res = df.copy()
    res[f"{column}_missing"] = res[column].isna().astype("boolean")
    return res, int(res[f"{column}_missing"].sum())

concernant allergens, celui ci nécéssite sa propre méthode, étant donné que les chaines vides ne veulent pas dire la même chose que sur les autres colonnes : 

In [ ]:
def flag_allergens(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """Allergènes : « vide » sans ingrédients = inconnu (NA) ; avec ingrédients = aucun allergène déclaré."""
    res = df.copy()
    allergens = res["allergens"].astype("string")

    # Inconnu : NULL d'origine, ou chaîne vide alors que les ingrédients sont absents
    empty_no_ingredients = allergens.eq("").fillna(False) & res["ingredients_text"].isna()
    unknown = allergens.isna() | empty_no_ingredients

    res["allergens"] = allergens.mask(unknown)
    res["allergens_missing"] = unknown.astype("boolean")
    return res, {
        "unknown_to_na": int(unknown.sum()),
        "none_declared": int(allergens.eq("").fillna(False).sum()),
    }

la méthode du cours faisait une conversion des valeurs en kj de la colonne vers les kcal si ceux ci étais manquant, hors dans le parquet ce n'est pas le cas la colonne energy-kcal contient uniquement les valeurs en kcal, de ce fait on modifie la create table pour creer une nouvelle colonne energy_kj_100g qui sera la fusion des deux colonnes du parquet : energy-kj et energy

In [15]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE kcal IS NULL) AS kcal_manquants,
        COUNT(*) FILTER (WHERE kcal IS NULL AND kj IS NOT NULL) AS recuperables_kj
    FROM (
        SELECT
            list_filter(t.nutriments, x -> x.name = 'energy-kcal')[1]."100g" AS kcal,
            COALESCE(
                list_filter(t.nutriments, x -> x.name = 'energy-kj')[1]."100g",
                list_filter(t.nutriments, x -> x.name = 'energy')[1]."100g"
            ) AS kj
        FROM '../../data/food.parquet' AS t
    )
""")

┌────────────────┬─────────────────┐
│ kcal_manquants │ recuperables_kj │
│     int64      │      int64      │
├────────────────┼─────────────────┤
│        1291656 │            8011 │
└────────────────┴─────────────────┘

la valeur 900 * 4184 est la conversion en kj pour le maximum possible pour 100gr ( 100 gr de lipides = 900kcal )

In [16]:
def impute_energy_business(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """2. Énergie : d'abord les kJ (exact), ensuite 4/4/9 (approximation), jamais une moyenne."""
    res = df.copy()
    kcal = res["energy_kcal_100g"]
    kj = res["energy_kj_100g"]

    # Étape exacte : kJ / 4,184 (hors bornes déjà neutralisées par RANGES)
    from_kj = kcal.isna() & kj.notna()
    kcal = kcal.mask(from_kj, (kj / 4.184).round(1))

    # Étape approximative : 4/4/9, uniquement si les trois macros sont présentes
    calc = 4 * res["carbohydrates_100g"] + 4 * res["proteins_100g"] + 9 * res["fat_100g"]
    from_macros = kcal.isna() & calc.between(0, 900)
    kcal = kcal.mask(from_macros, calc.round(1))

    res["energy_kcal_100g"] = kcal
    return res, {
        "from_kj": int(from_kj.sum()),
        "from_macros": int(from_macros.sum()),
        "remaining": int(kcal.isna().sum()),
    }

on passe a la méthode imputer_sel_sodium :

In [17]:
def impute_salt_sodium(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """3. Sel et sodium : relation exacte dans les deux sens, valeurs plausibles uniquement."""
    res = df.copy()
    salt, sodium = res["salt_100g"], res["sodium_100g"]

    # Hors bornes déjà neutralisées par RANGES : notna() suffit pour une valeur plausible
    salt_ok = salt.isna() & sodium.notna()
    sodium_ok = sodium.isna() & salt.notna()

    res["salt_100g"] = salt.mask(salt_ok, sodium * 2.5)
    res["sodium_100g"] = sodium.mask(sodium_ok, salt / 2.5)

    return res, {
        "salt_from_sodium": int(salt_ok.sum()),
        "sodium_from_salt": int(sodium_ok.sum()),
        "remaining": int(res["salt_100g"].isna().sum()),
    }

Methode pour l'ajout du drapeau sur les colonnes varchar

In [ ]:
def flag_missing(df: pd.DataFrame, column: str) -> tuple[pd.DataFrame, int]:
    """ajout du drapeau missing sur les colonnes varchar ne demandant pas d'imputation"""
    res = df.copy()
    res[f"{column}_missing"] = res[column].isna().astype("boolean")
    return res, int(res[f"{column}_missing"].sum())

Methode pour l'ajout du drapeau sur la colonne categories_tag

In [ ]:
def flag_unclassifiable(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """Produit sans catégorie : inclassable, exclu de la substitution."""
    res = df.copy()
    empty = res["categories_tags"].isna() | (res["categories_tags"].str.len() == 0)
    res["unclassifiable"] = empty.astype("boolean")
    return res, {"unclassifiable": int(empty.sum())}

problème si la colonne brand contient des espaces ou des chaines vide, isna ne les détectera pas en tant que NULL, on vérifie

In [19]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE brands IS NULL) AS nulls,
        COUNT(*) FILTER (WHERE brands = '') AS empty_strings,
        COUNT(*) FILTER (WHERE brands <> '' AND trim(brands) = '') AS spaces_only
    FROM products_nutrition
""").show()

┌─────────┬───────────────┬─────────────┐
│  nulls  │ empty_strings │ spaces_only │
│  int64  │     int64     │    int64    │
├─────────┼───────────────┼─────────────┤
│ 1703767 │             0 │           0 │
└─────────┴───────────────┴─────────────┘



même chose pour ingredients_text

In [32]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE ingredients_text IS NULL) AS nulls,
        COUNT(*) FILTER (WHERE ingredients_text = '') AS empty_strings,
        COUNT(*) FILTER (WHERE ingredients_text <> '' AND trim(ingredients_text) = '') AS spaces_only
    FROM products_nutrition
""").show()

┌─────────┬───────────────┬─────────────┐
│  nulls  │ empty_strings │ spaces_only │
│  int64  │     int64     │    int64    │
├─────────┼───────────────┼─────────────┤
│ 3339871 │             0 │           0 │
└─────────┴───────────────┴─────────────┘



il ya effectivement des chaines vides sur brands, je modifie la colonne brands sur le duckdb pour les transformer en null

In [25]:
def median_by_food_group_after_split(train: pd.DataFrame, test: pd.DataFrame, column: str) -> tuple[pd.Series, pd.Series]:
    """L'imputation statistique se calcule sur le train et s'applique au test : jamais l'inverse."""
    medians = train.groupby("food_groups_tag_1")[column].median()
    fallback = train[column].median()
    fill = lambda d: d[column].fillna(d["food_groups_tag_1"].map(medians)).fillna(fallback)  # noqa: E731
    return fill(train), fill(test)

ok je vais désormais rédiger les méthodes pour les colonnes qui ne sont pas dans l'exercice et celles dont avais établis de nouvelles règles, on avais établis la règle suivante si fat_100g < 0.5 alors saturated_fat_100g = 0, je lance une requete pour voir combien de lignes nutriment ne respecte pas ces règles

In [26]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE fat_100g < 0.5) AS fat_below_threshold,
        COUNT(*) FILTER (WHERE fat_100g < 0.5 AND saturated_fat_100g IS NULL) AS sat_null,
        COUNT(*) FILTER (WHERE fat_100g < 0.5 AND saturated_fat_100g = 0) AS sat_zero,
        COUNT(*) FILTER (WHERE fat_100g < 0.5 AND saturated_fat_100g > 0) AS sat_positive
    FROM products_nutrition
""").show()

┌─────────────────────┬──────────┬──────────┬──────────────┐
│ fat_below_threshold │ sat_null │ sat_zero │ sat_positive │
│        int64        │  int64   │  int64   │    int64     │
├─────────────────────┼──────────┼──────────┼──────────────┤
│              637327 │   105620 │   440532 │        91172 │
└─────────────────────┴──────────┴──────────┴──────────────┘



on a donc 91172 cas qui ne respecte pas ces règles, il faut qu'on les inspecte

In [27]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE saturated_fat_100g > fat_100g) AS sat_above_fat,
    FROM products_nutrition
    WHERE fat_100g < 0.5
""").show()

┌───────────────┐
│ sat_above_fat │
│     int64     │
├───────────────┤
│          2117 │
└───────────────┘



Ici 2117 valeurs qui sont supérieur a fat (incohérence), on dois les neutraliser dans la méthode

In [28]:
def apply_saturated_fat_rule(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """Graisses saturées : bornées par le fat, et nulles si fat < 0,5 g."""
    res = df.copy()
    fat, sat = res["fat_100g"], res["saturated_fat_100g"]

    # Règle métier : NA d'origine et fat < 0,5 -> saturées = 0
    to_zero = sat.isna() & (fat < 0.5)

    # Incohérence de paire : saturées > fat -> NA (hors bornes déjà gérées par RANGES)
    invalid = sat.notna() & (sat > fat)
    sat = sat.mask(invalid)

    # Drapeau : NA d'origine + valeurs hors bornes (RANGES) + incohérences, avant le passage à 0
    res["saturated_fat_100g_missing"] = sat.isna().astype("boolean")

    res["saturated_fat_100g"] = sat.mask(to_zero, 0)
    return res, {
        "set_to_zero": int(to_zero.sum()),
        "invalid_to_na": int(invalid.sum()),
    }

on avais établis une regle de reconsitution du fat dans le tableau : métier fat = (énergie − 4 × protéines − 4 × glucides) / 9 possible si les 3 sont présentes, on va vérifier si il y a des lignes ou cette règle serais utile

In [ ]:
# 1. Où la reconstitution est possible et plausible
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE fat_100g IS NULL) AS fat_null,
        COUNT(*) FILTER (
            WHERE fat_100g IS NULL
              AND energy_kcal_100g BETWEEN 0 AND 900
              AND proteins_100g BETWEEN 0 AND 100
              AND carbohydrates_100g BETWEEN 0 AND 100
        ) AS reconstructible,
        COUNT(*) FILTER (
            WHERE fat_100g IS NULL
              AND energy_kcal_100g BETWEEN 0 AND 900
              AND proteins_100g BETWEEN 0 AND 100
              AND carbohydrates_100g BETWEEN 0 AND 100
              AND (energy_kcal_100g - 4 * proteins_100g - 4 * carbohydrates_100g) / 9 BETWEEN 0 AND 100
        ) AS plausible
    FROM products_nutrition
""").show()

┌──────────┬─────────────────┬───────────┐
│ fat_null │ reconstructible │ plausible │
│  int64   │      int64      │   int64   │
├──────────┼─────────────────┼───────────┤
│  1321219 │           10581 │      7619 │
└──────────┴─────────────────┴───────────┘



In [ ]:
# 2. Fiabilité de la formule là où le fat est connu
con.sql("""
    SELECT
        COUNT(*) AS n,
        median(abs(fat_100g - (energy_kcal_100g - 4 * proteins_100g - 4 * carbohydrates_100g) / 9)) AS median_abs_error,
        COUNT(*) FILTER (
            WHERE abs(fat_100g - (energy_kcal_100g - 4 * proteins_100g - 4 * carbohydrates_100g) / 9) <= 1
        ) * 100.0 / COUNT(*) AS pct_within_1g
    FROM products_nutrition
    WHERE fat_100g BETWEEN 0 AND 100
      AND energy_kcal_100g BETWEEN 0 AND 900
      AND proteins_100g BETWEEN 0 AND 100
      AND carbohydrates_100g BETWEEN 0 AND 100
""").show()

┌─────────┬──────────────────┬───────────────────┐
│    n    │ median_abs_error │   pct_within_1g   │
│  int64  │      float       │      double       │
├─────────┼──────────────────┼───────────────────┤
│ 3254657 │        0.4000001 │ 73.51693281350386 │
└─────────┴──────────────────┴───────────────────┘



on drop la règle métier établi ici, le gain est faible en plus pour des approximations

on avais aussi établi une régle métier qui disais que les glucides doivent être forcément supérieur au sucre ( vu que les sucres font partie des glucides ) on vérifie, mais je vérifie aussi que ces lignes ne sont pas sur des glucides aberrant

In [29]:
con.sql("""
    SELECT
        COUNT(*) FILTER (WHERE carbohydrates_100g NOT BETWEEN 0 AND 100) AS carbs_invalid,
        COUNT(*) FILTER (
            WHERE carbohydrates_100g NOT BETWEEN 0 AND 100
              AND sugars_100g IS NOT NULL
        ) AS carbs_invalid_with_sugars,
        COUNT(*) FILTER (
            WHERE carbohydrates_100g NOT BETWEEN 0 AND 100
              AND sugars_100g > carbohydrates_100g
        ) AS sugars_above_invalid_carbs,
        COUNT(*) FILTER (
            WHERE carbohydrates_100g NOT BETWEEN 0 AND 100
              AND sugars_100g BETWEEN 0 AND 100
        ) AS sugars_valid_with_invalid_carbs
    FROM products_nutrition
""").show()

┌───────────────┬───────────────────────────┬────────────────────────────┬─────────────────────────────────┐
│ carbs_invalid │ carbs_invalid_with_sugars │ sugars_above_invalid_carbs │ sugars_valid_with_invalid_carbs │
│     int64     │           int64           │           int64            │              int64              │
├───────────────┼───────────────────────────┼────────────────────────────┼─────────────────────────────────┤
│         14335 │                     13887 │                         26 │                            7475 │
└───────────────┴───────────────────────────┴────────────────────────────┴─────────────────────────────────┘



Je créer la méthode pour neutraliser les sucres qui ne suivent pas cette règle, il faudra lancer la neutralisation des glucides avant de passer celle ci

In [30]:
def neutralize_sugars(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    """Sucres : supérieurs aux glucides -> NA, jamais imputés."""
    res = df.copy()
    sugars, carbs = res["sugars_100g"], res["carbohydrates_100g"]

    # Incohérence de paire : sucres > glucides (glucides déjà neutralisés par RANGES)
    above_carbs = sugars.notna() & carbs.notna() & (sugars > carbs)
    sugars = sugars.mask(above_carbs)

    # Drapeau : NA d'origine + valeurs hors bornes (RANGES) + incohérences de paire
    res["sugars_100g_missing"] = sugars.isna().astype("boolean")

    res["sugars_100g"] = sugars
    return res, {
        "above_carbs_to_na": int(above_carbs.sum()),
        "missing_total": int(sugars.isna().sum()),
    }

on passe a la méthode pour supprimer les lignes qui n'ont aucun nutriment clé, d'abord une requete pour vérifier le nombre de lignes concerné, on ne prend que les nutriment qui entre dans le calcul du nutriscore ici :

In [31]:
con.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (
            WHERE energy_kcal_100g IS NULL AND energy_kj_100g IS NULL
              AND saturated_fat_100g IS NULL AND sugars_100g IS NULL
              AND proteins_100g IS NULL AND salt_100g IS NULL
              AND sodium_100g IS NULL
        ) AS no_key_nutrient,
        ROUND(100.0 * COUNT(*) FILTER (
            WHERE energy_kcal_100g IS NULL AND energy_kj_100g IS NULL
              AND saturated_fat_100g IS NULL AND sugars_100g IS NULL
              AND proteins_100g IS NULL AND salt_100g IS NULL
              AND sodium_100g IS NULL
        ) / COUNT(*), 1) AS pct
    FROM products_nutrition
""").show()

┌─────────┬─────────────────┬────────┐
│  total  │ no_key_nutrient │  pct   │
│  int64  │      int64      │ double │
├─────────┼─────────────────┼────────┤
│ 4636471 │         1241528 │   26.8 │
└─────────┴─────────────────┴────────┘



maitenant on peux passer a la méthode 

In [ ]:
def drop_rows_without_key_nutrient(df: pd.DataFrame, columns: list[str] = KEY_NUTRIENTS) -> tuple[pd.DataFrame, dict[str, int]]:
    """Supprime les lignes dont tous les nutriments clés sont manquants."""
    mask = df[columns].isna().all(axis=1)
    res = df.loc[~mask].reset_index(drop=True)
    return res, {
        "rows_before": len(df),
        "rows_dropped": int(mask.sum()),
        "rows_after": len(res),
    }

on peux passer a la dernière méthode, celle qui remplacera les NA des colonnes qu'on avais décidé dans le tableau par une médiane

Toute nos méthodes pour les différentes colonnes sont faite, on écris donc le déroulé de nettoyage logique :

In [ ]:
# --- Constantes ---
KEY_NUTRIENTS = [
    "energy_kcal_100g", "energy_kj_100g", "saturated_fat_100g",
    "sugars_100g", "proteins_100g", "salt_100g", "sodium_100g",
]

RANGES = {
    # Colonnes avec drapeau posé par neutralize_range
    "fat_100g": (0, 100, True),
    "proteins_100g": (0, 100, True),
    "fiber_100g": (0, 100, True),
    "fruits_vegetables_legumes_100g": (0, 100, True),
    "carbohydrates_100g": (0, 100, True),

    # Colonnes dont le drapeau est posé dans leur méthode (après un contrôle entre colonnes)
    "sugars_100g": (0, 100, False),
    "saturated_fat_100g": (0, 100, False),

    # Colonnes sans drapeau
    "salt_100g": (0, 100, False),
    "sodium_100g": (0, 40, False),          # sodium ≈ 40 % de la masse du sel
    "energy_kcal_100g": (0, 900, False),    # 100 g de lipides purs = 900 kcal
    "energy_kj_100g": (0, 900 * 4.184, False),
}

# --- Définitions ---
# normalize_grade, neutralize_range, flag_missing, neutralize_carbohydrates,
# neutralize_sugars, apply_saturated_fat_rule, impute_salt_sodium,
# drop_rows_without_key_nutrient, impute_energy_business,
# median_by_aisle_after_split

# --- Déroulé du nettoyage ---

# 1. Cible : unknown / not-applicable -> NA
df, grade_report = normalize_grade(df)

# 2. Garde-fous + drapeaux (fat, protéines, fibres, fruits/légumes, kJ)
range_reports = {}
for col, (low, high, flag) in RANGES.items():
    df, range_reports[col] = neutralize_range(df, col, low, high, flag)

# 3. Drapeaux simples (colonnes sans garde-fou)
for col in ["brands", "ingredients_text", "nova_group", "product_name"]:
    df, n_missing = flag_missing(df, col)

# 3 bis. Drapeau allergens
df, allergens_report = flag_allergens(df)

# 4. Neutralisations et règles métier (drapeaux posés dans les méthodes)
df, sugars_report = neutralize_sugars(df)
df, sat_report = apply_saturated_fat_rule(df)      # après le garde-fou du fat
df, salt_report = impute_salt_sodium(df)

# 5. Suppression des lignes sans composante du score
df, drop_report = drop_rows_without_key_nutrient(df)

# 6. Énergie : garde-fou, kJ / 4,184, puis 4/4/9
df, energy_report = impute_energy_business(df)

# 6 bis. Drapeau énergie : lignes encore NA, donc celles que la médiane va remplir
df, n_energy_imputed = flag_missing(df, "energy_kcal_100g")

# 7. Split, puis médianes par food_groups_tag_1 calculées sur le train uniquement
# train, test = ...
for col in ["energy_kcal_100g", "fat_100g", "proteins_100g"]:
    train[col], test[col] = median_by_food_group_after_split(train, test, col)